# D-12 Data Preprocessing — Age-wise Migrants by Last Residence (Census 2011)

**File:** DS-0000-D12-MDDS.XLSX  
**What this dataset has:** For each state (destination), migrants from each last-residence state, broken down by **age group** (0–4, 5–9, 10–14 ... 80+) with Persons / Males / Females for each group.

---

### Preprocessing Plan

**Columns to DROP (Excel refs A, B, C, E, G, H, I, J):**
- A → Table Name
- B → State Code
- C → District Code
- E → Place of Enumeration (Total/Rural/Urban) — filter to Total first, then drop
- G → Last Residence TRU (Total/Rural/Urban) — filter to Total first, then drop
- H, I, J → Total Migrants (Persons, Males, Females) — aggregate column, not needed

**Columns to KEEP:**
- D → Area Name (destination state)
- F → Last Residence (origin state)
- K onwards → All Age Group columns (Persons + Males + Females for each age band)

**Rows to DROP:**
- Rows 6–329 in Excel (Area Name = INDIA) — we only want state-level data
- Any row where Place of Enumeration (col E) is Rural or Urban — keep Total only
- Any row where Last Residence TRU (col G) is Rural or Urban — keep Total only
- Summary rows inside LastResidence (e.g. 'Total', 'Total - Within State') — keep only actual state-to-state flows

---
## Step 0 — Install & Import Libraries

In [ ]:
# Install openpyxl if not already available (needed for reading .xlsx in Colab)
!pip install openpyxl --quiet

import pandas as pd
import numpy as np

print("Libraries loaded successfully.")

---
## Step 1 — Upload & Load the Raw Dataset

Upload `DS-0000-D12-MDDS.XLSX` using the Colab file uploader below, then run the cell.

In [ ]:
from google.colab import files

print("Please upload DS-0000-D12-MDDS.XLSX")
uploaded = files.upload()

In [ ]:
# Load the raw Excel file — no header, read everything as-is
df_raw = pd.read_excel('DS-0000-D12-MDDS.XLSX', header=None)

print(f"Raw shape  : {df_raw.shape}  ({df_raw.shape[0]} rows × {df_raw.shape[1]} columns)")
print()
print("First 6 rows (header block + first data row):")
df_raw.head(6)

---
## Step 2 — Understand the Raw Column Structure

The first 5 rows are multi-row headers. We extract them here to confirm column positions before any cleaning.

In [ ]:
# Display the 4 header rows (rows index 1–4) to understand column layout
print("=== HEADER ROWS (rows 1–4) ===")
header_display = df_raw.iloc[1:5, :].copy()
header_display.index = ['Row 1 (label)', 'Row 2 (age group)', 'Row 3 (P/M/F)', 'Row 4 (col number)']
print(header_display.to_string())
print()

# Quick column-by-column summary of what each position represents
print("=== COLUMN POSITION SUMMARY ===")
col_map = {
    0:  'A - Table Name',
    1:  'B - State Code',
    2:  'C - District Code',
    3:  'D - Area Name (destination state)',
    4:  'E - Place of Enumeration (Total/Rural/Urban)',
    5:  'F - Last Residence (origin state/place)',
    6:  'G - Last Residence TRU (Total/Rural/Urban)',
    7:  'H - Total Migrants Persons',
    8:  'I - Total Migrants Males',
    9:  'J - Total Migrants Females',
    10: 'K - Age 0-4 Persons',
    11: 'L - Age 0-4 Males',
    12: 'M - Age 0-4 Females',
    # ... continues in triplets through col 63
}
for col_idx, desc in col_map.items():
    print(f"  Col {col_idx:2d} : {desc}")
print(f"  Col 10–63 : Age group data (Persons/Males/Females for each of 18 age bands)")

---
## Step 3 — Extract Data Rows & Assign Working Column Names

Skip the 5-row header block (rows 0–4). Assign readable names to all 64 columns.

In [ ]:
# Skip the 5 header rows (indices 0–4), reset index
df = df_raw.iloc[5:].reset_index(drop=True)
print(f"Data rows after skipping header block: {len(df)}")

# ── Age group labels in order ──────────────────────────────────────────────
age_groups = [
    '0to4', '5to9', '10to14', '15to19', '20to24', '25to29',
    '30to34', '35to39', '40to44', '45to49', '50to54', '55to59',
    '60to64', '65to69', '70to74', '75to79', '80plus', 'AgeNS'
]

# ── Build full column name list ────────────────────────────────────────────
col_names = [
    'TableName',          # col 0  — A
    'StateCode',          # col 1  — B
    'DistrictCode',       # col 2  — C
    'AreaName',           # col 3  — D  ← KEEP
    'PlaceOfEnum_TRU',    # col 4  — E  ← filter to Total, then drop
    'LastResidence',      # col 5  — F  ← KEEP
    'LastRes_TRU',        # col 6  — G  ← filter to Total, then drop
    'TotalMig_Persons',   # col 7  — H  ← DROP
    'TotalMig_Males',     # col 8  — I  ← DROP
    'TotalMig_Females',   # col 9  — J  ← DROP
]

# Add age group columns: Persons / Males / Females for each age band
for ag in age_groups:
    col_names.append(f'Persons_{ag}')
    col_names.append(f'Males_{ag}')
    col_names.append(f'Females_{ag}')

# Assign names
df.columns = col_names

print(f"\nTotal columns assigned : {len(col_names)}")
print(f"Column names           : {col_names}")

---
## Step 4 — Row Cleaning

### 4.1 — Drop all INDIA rows (Excel rows 6–329)
We only need state-level data. All rows where `AreaName == 'INDIA'` are removed.

In [ ]:
rows_before = len(df)

# Remove all rows where AreaName is INDIA (aggregate national rows)
df = df[df['AreaName'].astype(str).str.strip().str.upper() != 'INDIA'].reset_index(drop=True)

rows_after = len(df)
print(f"Rows before : {rows_before}")
print(f"Rows dropped (INDIA) : {rows_before - rows_after}")
print(f"Rows after  : {rows_after}")
print()
print("Unique AreaName values remaining (first 5):")
print(df['AreaName'].unique()[:5])

### 4.2 — Filter `LastRes_TRU` (col G) → Keep 'Total' only, then drop column

In [ ]:
rows_before = len(df)

# Check what values exist in this column
print("Values in LastRes_TRU (col G):")
print(df['LastRes_TRU'].value_counts())
print()

# Keep only 'Total' rows — removes Rural and Urban variants
df = df[df['LastRes_TRU'].astype(str).str.strip() == 'Total'].reset_index(drop=True)

# Now drop the column — no longer needed
df = df.drop(columns=['LastRes_TRU'])

print(f"Rows before : {rows_before}")
print(f"Rows dropped (Rural/Urban LastRes_TRU) : {rows_before - len(df)}")
print(f"Rows after  : {len(df)}")

### 4.3 — Filter `PlaceOfEnum_TRU` (col E) → Keep 'Total' only, then drop column

In [ ]:
rows_before = len(df)

# Check values
print("Values in PlaceOfEnum_TRU (col E):")
print(df['PlaceOfEnum_TRU'].value_counts())
print()

# Keep only 'Total' rows — removes Rural and Urban variants
df = df[df['PlaceOfEnum_TRU'].astype(str).str.strip() == 'Total'].reset_index(drop=True)

# Drop the column — no longer needed
df = df.drop(columns=['PlaceOfEnum_TRU'])

print(f"Rows before : {rows_before}")
print(f"Rows dropped (Rural/Urban PlaceOfEnum) : {rows_before - len(df)}")
print(f"Rows after  : {len(df)}")

---
## Step 5 — Column Cleaning

### 5.1 — Drop redundant identifier columns (A, B, C) and aggregate total columns (H, I, J)

In [ ]:
cols_to_drop = ['TableName', 'StateCode', 'DistrictCode',
                'TotalMig_Persons', 'TotalMig_Males', 'TotalMig_Females']

df = df.drop(columns=cols_to_drop)

print(f"Columns after drop : {len(df.columns)}")
print(f"Remaining columns  : {df.columns.tolist()}")

### 5.2 — Clean `AreaName` column

Raw values look like `'State - JAMMU & KASHMIR (01)'`. We strip the prefix and suffix to get clean state names matching the D01/D02 format.

In [ ]:
import re

def clean_area_name(name):
    """Remove 'State - ' prefix and ' (XX)' state code suffix. Title-case the result."""
    name = str(name).strip()
    # Remove 'State - ' prefix
    name = re.sub(r'^State\s*-\s*', '', name, flags=re.IGNORECASE)
    # Remove trailing state code like ' (01)'
    name = re.sub(r'\s*\(\d+\)\s*$', '', name)
    # Title case
    name = name.strip().title()
    # Fix known abbreviations that title-case breaks
    name = name.replace('Nct Of Delhi', 'NCT of Delhi')
    name = name.replace('A & N Islands', 'Andaman & Nicobar Islands')
    name = name.replace('D & N Haveli', 'Dadra & Nagar Haveli')
    name = name.replace('D&N Haveli', 'Dadra & Nagar Haveli')
    return name

df['AreaName'] = df['AreaName'].apply(clean_area_name)

print("Unique AreaName values after cleaning:")
for name in sorted(df['AreaName'].unique()):
    print(f"  {name}")

### 5.3 — Clean `LastResidence` column

Strip whitespace, fix title-casing for consistency with D01/D02 naming.

In [ ]:
df['LastResidence'] = df['LastResidence'].astype(str).str.strip()

print("Unique LastResidence values:")
for val in sorted(df['LastResidence'].unique()):
    print(f"  {val}")

---
## Step 6 — Remove Summary / Aggregate Rows from LastResidence

Rows where `LastResidence` is a summary label (e.g. 'Total', 'Total - Within State') inflate counts. We remove them and keep only actual state-to-state flows.

In [ ]:
rows_before = len(df)

# These are summary/aggregate LastResidence labels — not actual states
summary_labels = [
    'Total',
    'Total - Within State',
    'Last Residence Within State',
    'Other States/Uts',
    'Other States/UTs',
    'Nepal',
    'Bangladesh',
    'Pakistan',
    'Other Countries',
    'Abroad',
    'Outside India',
    'Unclassifiable',
    'Place Of Birth Not Reported',
    'Last Residence Not Reported',
]

# Normalise for comparison
summary_labels_lower = [s.lower().strip() for s in summary_labels]
df = df[~df['LastResidence'].str.lower().str.strip().isin(summary_labels_lower)].reset_index(drop=True)

print(f"Rows before : {rows_before}")
print(f"Rows dropped (summary LastResidence labels) : {rows_before - len(df)}")
print(f"Rows after  : {len(df)}")
print()
print("Remaining unique LastResidence values:")
for val in sorted(df['LastResidence'].unique()):
    print(f"  {val}")

---
## Step 7 — Convert All Numeric Columns to Integer

Age group columns may have been read as floats or objects. Convert all to integer, replacing any NaN with 0.

In [ ]:
# All columns except AreaName and LastResidence are numeric
numeric_cols = [c for c in df.columns if c not in ('AreaName', 'LastResidence')]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

print(f"Numeric columns converted : {len(numeric_cols)}")
print(f"Any remaining nulls       : {df.isnull().sum().sum()}")
print()
print(df.dtypes)

---
## Step 8 — Final Validation & Shape Check

In [ ]:
print("=" * 55)
print("  FINAL CLEANED DATASET SUMMARY")
print("=" * 55)
print(f"Shape          : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Null values    : {df.isnull().sum().sum()}")
print(f"Unique states (AreaName)      : {df['AreaName'].nunique()}")
print(f"Unique origins (LastResidence): {df['LastResidence'].nunique()}")
print()
print("Columns:")
for i, col in enumerate(df.columns):
    print(f"  {i+1:2d}. {col}")
print()
print("Sample rows:")
df.head(10)

In [ ]:
# Quick sanity check — total persons across all age groups should roughly match
# known census totals. Check one state as a spot-check.
sample_state = df['AreaName'].iloc[0]
state_df = df[df['AreaName'] == sample_state]

persons_cols = [c for c in df.columns if c.startswith('Persons_')]
print(f"Spot check — {sample_state}")
print(f"  Rows                 : {len(state_df)}")
print(f"  Total Persons_0to4   : {state_df['Persons_0to4'].sum():,}")
print(f"  Total Persons_20to24 : {state_df['Persons_20to24'].sum():,}")
print(f"  Sum across all age groups (Persons): {state_df[persons_cols].sum().sum():,}")

---
## Step 9 — Export Cleaned Dataset

In [ ]:
output_filename = 'D12_cleaned.csv'
df.to_csv(output_filename, index=False)
print(f"Saved → {output_filename}")
print(f"Shape  : {df.shape}")
print(f"Size   : {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

In [ ]:
# Download the cleaned CSV to your local machine
from google.colab import files
files.download('D12_cleaned.csv')
print("Download triggered for D12_cleaned.csv")